In [1]:
import sys
sys.path.append("..")

import pandas as pd

from src.embeddings import NILCEmbeddings
from src.vectorstore import carregar_colecao
from src.llm import criar_llm
from src.pipeline import responder
from src.perguntas_teste import PERGUNTAS

store = carregar_colecao(NILCEmbeddings())
llm = criar_llm()

linhas, resultados = [], {}
for tipo, esperado, pergunta in PERGUNTAS:
    r = responder(pergunta, store, llm)
    if r.motivo == "erro_llm":
        veredito = "falha_llm"
    else:
        recusou = r.motivo is not None
        certo = (recusou and esperado == "recusar") or (not recusou and esperado == "responder")
        veredito = "ok" if certo else "ERRO"
    resultados[pergunta] = r
    linhas.append(dict(tipo=tipo, esperado=esperado, motivo=r.motivo or "-", veredito=veredito))
    print(f"{veredito:9} | {tipo:18} | {r.motivo or 'respondida':26} | {pergunta}")

tabela = pd.DataFrame(linhas)
print()
print(pd.crosstab(tabela["esperado"], tabela["veredito"]))

c:\dev\tech-challenge-fase4\venv\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.1) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
c:\dev\tech-challenge-fase4\venv\Lib\site-packages\langchain_google_genai\chat_models.py:49: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  from google.generativeai.caching import CachedContent  # type: ignore[import]


ok        | ampla              | respondida                 | Quais são os principais problemas relatados pelos clientes?
ok        | ampla              | respondida                 | Quais reclamações estão relacionadas à entrega?
ok        | ampla              | respondida                 | O que os clientes mais elogiam nos produtos?
ok        | ampla              | respondida                 | Quais problemas aparecem nas avaliações de clientes insatisfeitos?
ok        | ampla              | respondida                 | Existem padrões de reclamação sobre a qualidade dos produtos?
ok        | especifica         | respondida                 | Há reclamações sobre produtos que chegaram com a embalagem danificada?
ok        | especifica         | respondida                 | Há queixas sobre falta de resposta do atendimento?
ok        | especifica         | respondida                 | Clientes relatam dificuldade para trocar ou devolver produtos?
ok        | especifica_ausente | sem_

In [3]:
for (tipo, esperado, pergunta), linha in zip(PERGUNTAS, linhas):
    if linha["veredito"] == "ERRO":
        r = resultados[pergunta]
        print("=" * 100)
        print(f"[{tipo}] esperado: {esperado} | motivo: {r.motivo} | {pergunta}")
        print(r.resposta.texto)

o codigo falhou pois atingiu a cota do gemini, vamos testar outro modelo

trocando o modelo, o teste executou e trouxe o esperado, as camadas de segurança funcionam e o llm esta respondendo somente o esperado.